# Week 5 — Customer Segmentation with K-Means

**Theme:** Unsupervised learning II — clustering (k-means)

A mall wants to understand its customers so it can target promotions better.
Nobody has labeled customers as "budget shopper" or "big spender" — but if we
plot income vs. spending, natural groups might just... appear. That's what
**clustering** does: find groups of similar points with no labels given.

**K-Means algorithm, in one paragraph:** pick `k` random cluster centers ->
assign every point to its nearest center -> move each center to the average of
its assigned points -> repeat until centers stop moving.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## 1. Create a synthetic customer dataset

We simulate 5 realistic customer segments (this also means we secretly *know*
the "true" answer, which is useful for checking whether K-Means finds it).

In [ ]:
rng = np.random.default_rng(42)

segments = [
    # (mean_income_k$, mean_spending_score, n_customers)
    (25, 20, 40),   # low income, low spending
    (25, 80, 40),   # low income, high spending (impulsive)
    (55, 50, 40),   # mid income, mid spending
    (85, 20, 40),   # high income, low spending (frugal)
    (85, 85, 40),   # high income, high spending
]

incomes, spending = [], []
for mean_income, mean_spend, n in segments:
    incomes.append(rng.normal(mean_income, 5, n))
    spending.append(rng.normal(mean_spend, 8, n))

income = np.concatenate(incomes)
spending_score = np.concatenate(spending)
X = np.column_stack([income, spending_score])
print("Customers:", X.shape[0])

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], alpha=0.6, edgecolor="k")
plt.title("Customers: Annual Income vs. Spending Score (unlabeled)")
plt.xlabel("Annual income ($k)")
plt.ylabel("Spending score (1-100)")
plt.show()

## 2. How many clusters? The elbow method

We don't know `k` in advance. We try several values and plot "inertia" (how
tightly packed each cluster is) — look for the "elbow" where adding more
clusters stops helping much.

In [ ]:
inertias = []
k_range = range(1, 10)
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(list(k_range), inertias, marker="o")
plt.title("Elbow Method")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia (within-cluster sum of squares)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Silhouette score is another way to pick k: higher is better (max 1.0)
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    score = silhouette_score(X, km.labels_)
    print(f"k={k}: silhouette score = {score:.3f}")

## 3. Run K-Means with the chosen k

Both the elbow plot and the silhouette scores should point toward **k=5** —
which matches how we generated the data.

In [ ]:
k = 5
kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(X)

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=cluster_labels, cmap="tab10", alpha=0.7, edgecolor="k")
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            c="red", marker="X", s=250, edgecolor="black", label="cluster center")
plt.title(f"K-Means Clustering (k={k})")
plt.xlabel("Annual income ($k)")
plt.ylabel("Spending score (1-100)")
plt.legend()
plt.show()

## 4. Interpret the segments

Turn the raw cluster centers into a business-readable table.

In [ ]:
import pandas as pd

summary = pd.DataFrame(kmeans.cluster_centers_, columns=["avg_income_k$", "avg_spending_score"])
summary["n_customers"] = pd.Series(cluster_labels).value_counts().sort_index().values
summary.index.name = "cluster"
summary

## Try it yourself

1. **Pick the wrong k.** Re-run K-Means with `k=2` and `k=8` and re-plot — how
   does the clustering change? Which one looks "wrong" given the elbow plot?
2. **Name the segments.** Based on the summary table, write a one-word label
   for each cluster (e.g. "frugal high earners", "impulsive low earners").
3. **Add a third feature.** Simulate a `visits_per_month` column and re-run
   K-Means on all 3 features — plot 2 of the 3 dimensions to visualize.
4. **Compare to Week 4.** Both PCA and K-Means are unsupervised — what's the
   key difference in what each one is *for*? (Hint: one compresses features,
   the other groups examples.)

---
## 🎯 캡스톤: 스터디 그룹 매칭 서비스

가상의 학생 120명의 "선호 공부 시작 시각 / 선호 그룹 인원 / 몰입 강도" 더미 데이터를 드립니다. 위에서 배운 K-Means로 이들을 몇 개의 "스터디 성향 그룹"으로 나눠보고, **여러분 자신의 선호도**를 입력해서 어떤 그룹에 가장 잘 맞는지 찾아보세요.

**확장 아이디어:** 실제 스터디 모집 설문(선호 시간대, 인원, 강도 등)을 만들어 친구들 응답을 모으면, 이 코드로 진짜 "스터디 그룹 자동 매칭"을 만들 수 있습니다.

In [ ]:
# 더미 데이터 생성 (실행만 하면 됩니다)
import pandas as pd
rng = np.random.default_rng(21)

# 4가지 스터디 성향 (선호 시작 시각 0-23시, 선호 인원 1-8명, 몰입 강도 1-10)
archetypes = {
    "새벽 집중파":      (6, 2, 9),
    "카공족(카페 공부)": (14, 4, 5),
    "왁자지껄 그룹형":   (16, 7, 3),
    "밤샘 벼락치기형":   (23, 1, 8),
}

rows, true_type = [], []
for name, (hour, size, intensity) in archetypes.items():
    n = 30
    hours = np.clip(rng.normal(hour, 2, n), 0, 23)
    sizes = np.clip(rng.normal(size, 1.2, n), 1, 8)
    intensities = np.clip(rng.normal(intensity, 1.5, n), 1, 10)
    rows.append(np.column_stack([hours, sizes, intensities]))
    true_type += [name] * n

group_prefs_df = pd.DataFrame(np.vstack(rows), columns=["preferred_hour", "group_size_pref", "focus_intensity"])
group_prefs_df["true_type"] = true_type  # 실제 서비스라면 이런 라벨은 없습니다 -- 확인용으로만 사용
group_prefs_df.head()

### 여러분의 과제

1. `group_prefs_df[["preferred_hour", "group_size_pref", "focus_intensity"]]`에 `KMeans(n_clusters=4)`를 학습시키세요. (원한다면 Week 5 본문처럼 elbow method로 먼저 k를 확인해봐도 좋습니다.)
2. 각 클러스터의 평균 `preferred_hour` / `group_size_pref` / `focus_intensity`를 표로 출력해서, 클러스터마다 어떤 "스터디 성향"인지 이름을 붙여보세요.
3. 아래 `my_prefs`에 여러분 자신의 선호도를 입력하고, 학습된 `kmeans.predict()`로 어느 클러스터에 속하는지 확인하세요.
4. 여러분과 같은 클러스터에 속한 가상 학생이 몇 명인지, 그리고 (선택) 3개 특성 중 2개를 골라 산점도에 여러분의 위치를 겹쳐 그려보세요.

In [ ]:
# TODO 1: KMeans(n_clusters=4)를 학습시키세요.


# TODO 2: 각 클러스터의 평균 특성을 표로 출력하고, 클러스터별로 이름을 붙여보세요.


# TODO 3: 나의 선호도를 입력하고 어느 클러스터에 속하는지 예측해보세요.
my_prefs = {
    "preferred_hour": None,     # 예: 20 (밤 8시)
    "group_size_pref": None,    # 예: 3
    "focus_intensity": None,    # 예: 7
}

# TODO 4: 같은 클러스터의 학생 수를 세어보고, 산점도에 나의 위치를 겹쳐 그려보세요.